## Create Table to summarize results

This notebook creates a table with the normalized NLLHs for different models. It expects to find pickled predictions for 10 folds with the following filenames:

 * ```<scenario_name>.floc.<dist_name>.<fold>.50.<model>.pkl```

`eval_model.py` automatically creates such files. 

**NOTE:** The first cell contains some variables that need to be adjusted. Evaluating all scenarios might take a while.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import tabulate

#% matplotlib inline

sys.path.append("../")
from helper_DistNet import load_data_continuous, preprocess

mean_exp = []  #["rf", "nn"]

# 1) Select the distribution you want to fit 
floc_exp = ["lognormal_distfit", "lognormal_distfit.rf2", "lognormal_distfit.rf", "lognormal_nn", "lognormal_xgb"]
distribution_hndl = scst.distributions.lognorm
# floc_exp = ["expon_distfit", "expon_distfit.rf2", "expon_distfit.rf", "expon_nn", "expon_xgb"]
# distribution_hndl = scst.distributions.expon
# floc_exp = ["invgauss_distfit", "invgauss_distfit.rf2", "invgauss_distfit.rf", "invgauss_nn", "invgauss_xgb"]
# distribution_hndl = scst.distributions.invgauss
# floc_exp = ["weibull_distfit", "weibull_distfit.rf2", "weibull_distfit.rf", "weibull_nn", "weibull_xgb"]
# distribution_hndl = scst.distributions.weibull_min

# 2) Change the following to existing paths
save_dir_output = "../results_all"
os.makedirs(save_dir_output, exist_ok=True)

output_dir = "../scripts_DistNet/TEST_50/"
save_dir = "./results_eval"

output_dir_xgb = "../../../Implementation_XGBoost/Continuous-master/scripts_DistNet/TEST_50_SMAC_NAS_100/"
save_dir_xgb = "../../../Implementation_XGBoost/Continuous-master/notebooks/results_eval_SMAC_NAS_100"

# 3) Select whether to compute results for all scenarios or just a single scenario
SCENARIOS = ["DE", "DiagonalCMA", "modcma", "modde", "NaiveIsoEMNA", "NGOpt14", "NGOpt38", "OnePlusOne", "PSO", "RandomSearch", "RCobyla"]
DIMS =  [5, 20]
BUDGETS = [100, 250, 500, 1000] # , 2500, 5000, 10000]
RUNS = 50
SEED = 50
LOG_TARGET = False

load = True

In [ ]:
res_dict = dict()
list_dirs = [output_dir, output_dir_xgb]
for output_dir in list_dirs:
    for fl in os.listdir(output_dir):
        # if not f".{SEED}." in fl:
        #     # Select only one seed
        #     continue
        print(fl, output_dir)

        fl_path = os.path.join(output_dir, fl)
        
        if os.path.isdir(fl_path):
            continue
        
        if not "pkl" in fl_path:
            continue
        
        if "txt" in fl_path or "png" in fl_path:
            continue

        if not os.path.isfile(fl_path):
            fl_path = os.path.join(output_dir)

        with open(fl_path, "rb") as fh:
            pkl = pickle.load(fh)
            train_pred = np.array(pkl[0])
            val_pred = np.array(pkl[1])
            add_info = pkl[2]
            task = add_info["task"]
            model = add_info["model"]
            scenario = add_info["scenario"]
            dim = add_info["dim"]
            budget = add_info["budget"]
            fold = add_info["fold"]

            if "nn" in add_info["model"] and add_info["task"] == "floc":
                print(f"{scenario}_{dim}D_{budget}, {fold}, {add_info['loaded']}")
            # Transform the xgboostlss parameter predictions
            if "xgb" in add_info["model"]:
                    if 'expon' in model:
                        train_pred = 1/train_pred
                        val_pred = 1/val_pred
                    elif 'lognormal' in model:
                        # Flip the parameter order
                        train_pred = train_pred[:, ::-1]
                        val_pred = val_pred[:, ::-1]
                        # Convert log of location -> location
                        train_pred[:, 1] = np.exp(train_pred[:, 1])
                        val_pred[:, 1] = np.exp(val_pred[:, 1])
                    elif 'weibull' in model:
                        # Flip the parameter order
                        train_pred = train_pred[:, ::-1]
                        val_pred = val_pred[:, ::-1]
    
        
        SC = f"{scenario}_{dim}D_{budget}"
        if SC not in res_dict:
            res_dict[SC] = dict()

        if model not in floc_exp and model not in mean_exp:
            continue
        
        if task not in res_dict[SC]:
            res_dict[SC][task] = dict()
            
        if model not in res_dict[SC][task]:
            res_dict[SC][task][model] = dict()

        res_dict[SC][task][model][fold] = (train_pred, val_pred)


if len(SCENARIOS) == 0 or len(DIMS) == 0 or len(BUDGETS) == 0:
    SCENARIOS = [key.split("_")[0] for key in res_dict.keys()]
    DIMS = list(set([int(key.split("_")[1][:-1]) for key in res_dict.keys()]))
    BUDGETS = list(set([int(key.split("_")[2]) for key in res_dict.keys()]))

In [ ]:
for scen in sorted(SCENARIOS):
    for dim in sorted(DIMS):
        for budget in sorted(BUDGETS):
            SC = f"{scen}_{dim}D_{budget}"
            print("### %s" % SC)
            try:
                for model in sorted(res_dict[SC]["mean"]):
                    print("mean %20s" % model, len(res_dict[SC]["mean"][model]))
            except:
                pass
            for model in sorted(res_dict[SC]["floc"]):
                print("floc %20s" % model, len(res_dict[SC]["floc"][model]))

In [ ]:
nllh_dict = dict()
mean_dict = dict()
var_dict = dict()
if load:
    with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
        nllh_dict2 = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
        mean_dict2 = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
        var_dict2 = pickle.load(fh)
    print("LOADED")

    with open(save_dir_xgb + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
        nllh_dict = pickle.load(fh)
        for scen in nllh_dict.keys():
            if scen not in nllh_dict2.keys():
                continue
            nllh_dict[scen] = nllh_dict[scen] | nllh_dict2[scen]
        
    with open(save_dir_xgb + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
        mean_dict = pickle.load(fh)
        for scen in mean_dict.keys():
            if scen not in mean_dict2.keys():
                continue
            mean_dict[scen] = mean_dict[scen] | mean_dict2[scen]
        
    with open(save_dir_xgb + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
        var_dict = pickle.load(fh)
        for scen in var_dict.keys():
            if scen not in var_dict2.keys():
                continue
            var_dict[scen] = var_dict[scen] | var_dict2[scen]
    print("LOADED")
    
    with open(save_dir_output + "/" + distribution_hndl.name + ".nllh.pkl", "wb") as fh:
        pickle.dump(nllh_dict, fh)

    with open(save_dir_output + "/" + distribution_hndl.name + ".mean.pkl", "wb") as fh:
        pickle.dump(mean_dict, fh)

    with open(save_dir_output + "/" + distribution_hndl.name + ".var.pkl", "wb") as fh:
        pickle.dump(var_dict, fh)
    print("dumped to " + save_dir_output)

else:
    print("NOT LOADED")

In [ ]:
# Generate NLLH table

# nan means there are some folds missing
# inf means there was no results file found
tab_data = list()
header = ["sc"] + floc_exp
CAP_NUM = 1e20
for scen in sorted(nllh_dict.keys(), key=lambda x: int(x.split("_")[2])):
    dummy_array = np.ones([2, 2]) * np.inf
    tab_data.append([scen] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) 
                              if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
                              else np.inf
                              for m in header[1:]])
    tab_data.append(["", ] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) 
                              if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
                              else np.inf
                              for m in header[1:]])

a = tabulate.tabulate(tab_data, headers=header, floatfmt="5.3f")
print(a)

with open("%s/nllh_%s_all.txt" % (save_dir, distribution_hndl.name), "w") as fh:
    fh.write(a)

## Generate Latex tables

In [ ]:
model_name = {
    "lognormal_distfit": "fitted",
    "lognormal_distfit.rf2": "iRF",
    "lognormal_distfit.rf": "mRF",
    "lognormal_nn": "DistNet",
    "lognormal_xgb": "XGBoostLSS",

    "invgauss_distfit": "fitted",
    "invgauss_distfit.rf2": "iRF",
    "invgauss_distfit.rf": "mRF",
    "invgauss_nn": "DistNet",
    "invgauss_xgb": "XGBoostLSS",

    "weibull_distfit": "fitted",
    "weibull_distfit.rf2": "iRF",
    "weibull_distfit.rf": "mRF",
    "weibull_nn": "DistNet",
    "weibull_xgb": "XGBoostLSS",
}

algo_name = {
    "DE": "DE",
    "DiagonalCMA": "Diagonal CMA",
    "modcma": "Mod CMA",
    "modde": "L-SHADE",
    "NaiveIsoEMNA": "NaiveIsoEMNA",
    "NGOpt14": "NGOpt14",
    "NGOpt38": "NGOpt38",
    "OnePlusOne": "OnePlusOne",
    "PSO": "PSO",
    "RandomSearch": "Random Search",
    "RCobyla": "RCobyla",
}

for algo in SCENARIOS:
    # algo = algo[0] # Choose one algorithm to create table for
    # algo = "RandomSearch"
    tabular_data = list()
    TAB_SCENARIOS = list()
    CAP_NUM = 1e10

    for dim in DIMS:
        for budget in BUDGETS:
            TAB_SCENARIOS.append(f"{algo}_{dim}D_{budget}")

    # header = ["Dimension"] + ["Budget"] + [""] + floc_exp
    # header.insert(-1, "Bayes DistNet")
    header = [""] + floc_exp
    header_start_index = len(header) - len(floc_exp)
    for scen in TAB_SCENARIOS:
        dummy_array = np.ones([2, 2]) * np.inf
        tabular_data.append(
            # [f"{scen.split('_')[1]}", f"{scen.split('_')[2]}"] + 
            ["train"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) 
            if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
            else np.inf for m in header[header_start_index:]]
        )
        tabular_data.append(
            # ["", "" ] + 
            ["test"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) 
            if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
            else np.inf for m in header[header_start_index:]]
        )
    latex_header = header[:header_start_index] + [model_name[m] for m in floc_exp]
    # latex_header.insert(-1, "Bayes DistNet")

    latex_tabular = tabulate.tabulate(tabular_data, tablefmt="latex", floatfmt="5.3f", headers=latex_header)
    # print(latex_tabular)

    os.makedirs(f"{save_dir_output}/{floc_exp[0].split('_')[0]}/", exist_ok=True)
    with open(f"{save_dir_output}/{floc_exp[0].split('_')[0]}/{algo}.tex", "w") as fh:
        fh.write(latex_tabular)

## Generate Boxplots

In [ ]:
# Create dataframes for test data
import pandas as pd
import seaborn as sns
from matplotlib.ticker import LogFormatter, LogLocator, FuncFormatter

model_name = {
    "lognormal_distfit": "fitted",
    "lognormal_distfit.rf2": "iRF",
    "lognormal_distfit.rf": "mRF",
    "lognormal_nn": "DistNet",
    "lognormal_xgb": "XGB",

    "invgauss_distfit": "fitted",
    "invgauss_distfit.rf2": "iRF",
    "invgauss_distfit.rf": "mRF",
    "invgauss_nn": "DistNet",
    "invgauss_xgb": "XGB",

    "weibull_distfit": "fitted",
    "weibull_distfit.rf2": "iRF",
    "weibull_distfit.rf": "mRF",
    "weibull_nn": "DistNet",
    "weibull_xgb": "XGB",
}

algo_name = {
    "DE": "DE",
    "DiagonalCMA": "Diagonal CMA",
    "modcma": "Mod CMA",
    "modde": "L-SHADE",
    "NaiveIsoEMNA": "NaiveIsoEMNA",
    "NGOpt14": "NGOpt14",
    "NGOpt38": "NGOpt38",
    "OnePlusOne": "OnePlusOne",
    "PSO": "PSO",
    "RandomSearch": "Random Search",
    "RCobyla": "RCobyla",
}

df = pd.DataFrame(columns=['algo','model', 'dim', 'budget', 'NLLH'])
CAP_NUM = 1e10
for algo in SCENARIOS:
    for dim in DIMS:
        for budget in BUDGETS:
            SC = f"{algo}_{dim}D_{budget}"
            dummy_array = np.ones([2, 2]) * np.inf
            for model in floc_exp:
                nllh_vals = nllh_dict[SC].get(model, dummy_array)[:, 1]
                nllh_vals = [nllh if nllh < CAP_NUM else CAP_NUM for nllh in nllh_vals]
                for nllh in nllh_vals:
                    df = pd.concat([df, pd.DataFrame({'algo': [algo],
                                                      'model': model_name.get(model, model),
                                                      'dim': [dim], 
                                                      'budget': [budget], 
                                                      'NLLH': [nllh]})], ignore_index=True)


In [ ]:
# Choose one algorithm to create boxplot for
plt.rcParams.update({
    "font.size": 16,      # default text size
    "axes.titlesize": 20, # subplot titles
    "axes.labelsize": 16, # x/y labels
    "xtick.labelsize": 18,
    "ytick.labelsize": 16,
    "legend.fontsize": 16
})
os.makedirs("./plots/boxplots/", exist_ok=True)
num_ticks = 5

# for algo in SCENARIOS:
for algo in ["NGOpt38"]: # ["RCobyla",  "RandomSearch", "modde", "DE", "PSO"]:
    # Subset dataframe
    df_algo = df[(df["algo"] == algo) & (~df["model"].isin(["iRF","mRF"]))].copy() #  

    plt.figure(figsize=(8,5))
    ax = sns.boxplot(
        data=df_algo,
        x="model",
        y="NLLH",
        hue="dim",          # separate colors for dimensions
        palette="Set2"      # nice readable colors
    )
    
    ax.set_yscale("symlog", linthresh=1, linscale=1) # 

    # Set custom y-tick labels
    # ticks = ax.get_yticks()
    # labels = []
    # for i, t in enumerate(ticks):
    #     if i % 2 == 0 or t == 0:
    #         labels.append(t)
    # ax.set_yticks(labels)
    
    # Set custom y-tick labels (2nd option)
    ymin, ymax = ax.get_ylim()
    # start, end = ymin, ymax
    # ticks = np.linspace(start, end, num_ticks)
    # # ticks = np.insert(ticks, 0, ymin)
    # # ticks = np.append(ticks, ymax)
    # ax.set_yticks(ticks)
    # ax.set_yticklabels([f"{t:.2f}" for t in ticks])

    ticks = list(ax.get_yticks())
    ticks.insert(0, ymin)
    ticks.append(ymax)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{t:.2f}" for t in ticks])
    
    ax.minorticks_off()
    
    ax.set_ylabel("Test NLLH (log scale)")
    ax.set_xlabel("Model")
    ax.legend(title="Dimension", loc="upper left")
    # ax.set_title(f"{algo_name.get(algo, algo)} | {floc_exp[0].split('_')[0].upper()}")

    plt.tight_layout()
    plt.savefig(f"./plots/boxplots/{floc_exp[0].split('_')[0]}_{algo}.pdf", dpi=300)
    plt.close()
